In [ ]:
import os
os.environ.pop("PYTHONPATH", None)

import sys
sys.path = [p for p in sys.path if "/opt/software-current/" not in p]

import regex
print("regex:", regex.__file__)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../../data/unsplash/clip_train.parquet")
collections_df = pd.read_csv("../../data/unsplash/collections.tsv000", sep="\t")
collections_df["collection_id"] = collections_df["collection_id"].astype(str)

df["image_path"] = [f"../../data/unsplash/{i}" for i in df["image_path"]]

In [ ]:
# inject the 10 laptop twee images into df AND into a "laptop twee" board
inject_df = pd.read_csv("../../data/small/metadata.csv")
inj = inject_df.iloc[:10].reset_index(drop=True)

new_rows = pd.DataFrame({
    "photo_id": [f"laptop_twee_{i}" for i in range(len(inj))],
    "image_path": [f"../../data/small/{p}" for p in inj["image_path"]],
    "caption": inj["caption"].astype(str),
})
df = pd.concat([df, new_rows], ignore_index=True)

collections_df = pd.concat([collections_df, pd.DataFrame({
    "photo_id": new_rows["photo_id"],
    "collection_id": "laptop_twee",
    "collection_title": "laptop twee",
})], ignore_index=True)

print(len(df), "photos")

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
from sentence_transformers import SentenceTransformer
from PIL import Image

MODEL_NAME = "google/siglip2-base-patch16-224"
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
embed_processor = AutoProcessor.from_pretrained(MODEL_NAME)
st = SentenceTransformer("all-MiniLM-L6-v2", device=device)

In [ ]:
def embed_images_batch(image_paths, batch_size=1024):
    embeddings = []
    for start in range(0, len(image_paths), batch_size):
        images = []
        for path in image_paths[start:start + batch_size]:
            with Image.open(path) as img:
                images.append(img.convert("RGB"))
        inputs = embed_processor(images=images, return_tensors="pt").to(device)
        with torch.inference_mode():
            out = embed_model.get_image_features(**inputs)
        out = getattr(out, "pooler_output", out)
        embeddings.append(F.normalize(out.float(), dim=-1))
    return torch.cat(embeddings)


def embed_text_batch(texts, batch_size=1024):  # SigLIP text -> matches images
    embeddings = []
    for start in range(0, len(texts), batch_size):
        inputs = embed_processor(text=texts[start:start + batch_size], padding="max_length",
                                 max_length=64, truncation=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            out = embed_model.get_text_features(**inputs)
        out = getattr(out, "pooler_output", out)
        embeddings.append(F.normalize(out.float(), dim=-1))
    return torch.cat(embeddings)


def embed_text_batch_st(texts, batch_size=1024):  # sentence-transformer -> matches text
    return st.encode(texts, batch_size=batch_size, convert_to_tensor=True,
                     normalize_embeddings=True, show_progress_bar=True).float()

In [ ]:
# SigLIP image embeddings, one per photo (slow: run once)
X = embed_images_batch(df["image_path"].tolist())
print(X.shape)

In [ ]:
# lookups + one text embedding per board (never the comma-joined titles)
photo_to_idx = {p: i for i, p in enumerate(df["photo_id"])}
unique_collections = (collections_df[["collection_id", "collection_title"]]
                      .drop_duplicates("collection_id").reset_index(drop=True))
collection_to_idx = {c: i for i, c in enumerate(unique_collections["collection_id"])}

titles = unique_collections["collection_title"].fillna("").astype(str).str.lower().tolist()
B = embed_text_batch_st(titles)

# which photo is in which board
pc = collections_df[collections_df["photo_id"].isin(photo_to_idx)].drop_duplicates(["photo_id", "collection_id"])
pc_photo = torch.tensor(pc["photo_id"].map(photo_to_idx).values, device=device)
pc_board = torch.tensor(pc["collection_id"].map(collection_to_idx).values, device=device)
board_size = torch.bincount(pc_board, minlength=len(titles)).float()

print(len(titles), "boards,", len(pc), "photo-board links")

In [ ]:
# style graph: each image -> its 10 most similar images,
# with the "laptop" direction removed so style matters, not the object
l = embed_text_batch(["a photo of a laptop"])[0]
X_style = F.normalize(X - (X @ l)[:, None] * l, dim=-1)

nbrs, nbr_w = [], []
for s in range(0, len(X_style), 8192):
    sim = X_style[s:s + 8192] @ X_style.T
    rows = torch.arange(len(sim), device=device)
    sim[rows, rows + s] = -1  # no self loops
    w, j = sim.topk(10, dim=1)
    nbrs.append(j)
    nbr_w.append(w.clamp(min=0))
nbrs, nbr_w = torch.cat(nbrs), torch.cat(nbr_w)
nbr_w = nbr_w / nbr_w.sum(1, keepdim=True).clamp(min=1e-8)

In [ ]:
def search(query, t=0.75, tau=0.05, beta=0.8, iters=10):
    # 1. query text matches BOARDS only (soft); small boards count more
    q_board = embed_text_batch_st([query.lower()])[0]
    r = torch.sigmoid((B @ q_board - t) / tau)
    board_w = r / torch.log(2 + board_size)

    # 2. seed = sum over ALL of a photo's boards
    seed = torch.zeros(len(X), device=device).index_add_(0, pc_photo, board_w[pc_board])
    seed = seed / seed.max().clamp(min=1e-8)

    # 3. diffuse seed scores to visually similar images
    s = seed.clone()
    for _ in range(iters):
        s = (1 - beta) * seed + beta * (s[nbrs] * nbr_w).sum(1)

    # 4. direct SigLIP text->image as tiebreaker, turned down when a board matches strongly
    siglip = X @ embed_text_batch([query])[0]
    w_siglip = 1 - 0.8 * r.max()
    z = lambda v: (v - v.mean()) / (v.std() + 1e-6)
    score = z(s) + w_siglip * z(siglip)

    print(unique_collections.assign(r=r.cpu().numpy()).nlargest(5, "r")[["collection_title", "r"]])
    return df.assign(score=score.cpu().numpy(), seed=seed.cpu().numpy()).sort_values("score", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

for query in ["laptop twee", "car"]:
    res = search(query)
    fig, axes = plt.subplots(5, 5, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        row = res.iloc[i]
        ax.imshow(Image.open(row["image_path"]))
        ax.axis("off")
        ax.set_title(f"{i + 1}{' SEED' if row['seed'] > 0.1 else ''}", fontsize=8,
                     color="green" if str(row["photo_id"]).startswith("laptop_twee_") else "black")
    fig.suptitle(query)
    plt.tight_layout()
    plt.show()